# 10. Gem infrasound node coordinates from GPS logs

Summarizes the `gemconvert` GPS logs for the 20260403 service run with
`flovopy.ingest.gem.summarize_gem_gps` (which calls `gemlog.summarize_gps`), using the campaign
serial→SEED mapping, and compares the median GPS positions with the station table.

The GPS medians are a QC product, not survey-grade coordinates. The GPS start/end times also show
when each Gem was running.

Requires `gemlog` in the kernel's environment.

In [1]:
import numpy as np
import pandas as pd

import artemis2_config as cfg
from flovopy.ingest.gem import summarize_gem_gps

OUT_CSV = cfg.QC_DIR / 'gem_gps_coordinates.csv'
print('GPS logs :', cfg.GEM_GPS_DIR)
print('mapping  :', cfg.GEM_MAPPING_CSV)
print('output   :', OUT_CSV)

GPS logs : /Volumes/KSCGTdownld/20260403_service/10_gemconvert/gps
mapping  : /Users/glennthompson/Developer/KSCRocketSeismoHydrology/2026_ingest_service_runs/config/gem_mapping_artemis2_example.csv
output   : /Volumes/KSCGTdownld/20260403_service/30_qc/gem_gps_coordinates.csv


In [2]:
gps = summarize_gem_gps(cfg.GEM_GPS_DIR, OUT_CSV, mapping=cfg.GEM_MAPPING_CSV)
mapping = pd.read_csv(cfg.GEM_MAPPING_CSV, dtype=str)
gps['SN'] = gps['SN'].astype(int).astype(str)
gps = gps.merge(mapping.rename(columns={'serial': 'SN', 'id': 'seed_id'}), on='SN', how='left')
gps[['SN', 'seed_id', 'lat', 'lon', 'starttime', 'endtime', 'num_samples']]

1 of 21: 244


2 of 21: 246


3 of 21: 251


4 of 21: 254


5 of 21: 256


6 of 21: 273


... (12 outputs left out)

20 of 21: 366


21 of 21: 369


Reading file /var/folders/yj/6sjdgxm52t5d_p4w0lcrsrzr0000gp/T/tmpu2zqea83/station_info.txt
File has valid header, using that for column names


,SN,seed_id,lat,lon,starttime,endtime,num_samples
0,244,1R.B14..HDI,28.632276,-80.615963,2026-03-11T18:14:03.000000Z,2026-04-04T19:44:35.000000Z,46001
1,246,1R.B14..HDF,28.632270,-80.615983,2026-03-11T18:08:03.000000Z,2026-04-04T19:53:23.000000Z,45763
2,251,1R.B01..HDI,28.621368,-80.606851,2026-03-11T17:15:00.000000Z,2026-04-04T20:30:38.000000Z,45997
3,254,1R.B03..HDI,28.643006,-80.623604,2026-03-11T19:25:25.000000Z,2026-04-04T18:56:18.000000Z,48084
4,256,1R.B03..HDF,28.642982,-80.623608,2026-03-11T19:28:23.000000Z,2026-04-01T20:28:53.000000Z,40687
5,273,1R.B24LT..HDF,28.580586,-80.642901,2026-03-10T16:43:23.000000Z,2026-04-03T18:58:52.000000Z,47844
6,274,1R.B07.10.HDF,28.607186,-80.594139,2026-03-13T16:27:41.000000Z,2026-04-04T21:28:41.000000Z,43403
7,275,1R.B24..HDI,28.580653,-80.640844,2026-03-10T17:46:24.000000Z,2026-04-08T17:31:53.000000Z,55060
8,285,1R.B20..HDF,28.573620,-80.572249,2026-03-11T14:23:51.000000Z,2026-04-04T22:08:53.000000Z,46608
9,287,1R.B24RE..HDF,28.581152,-80.643291,2026-03-12T16:56:39.000000Z,2026-04-03T19:12:28.000000Z,43294


## Serials in the mapping without GPS logs, and Gems whose GPS stops before launch

In [3]:
no_gps = sorted(set(mapping['serial']) - set(gps['SN']), key=int)
print('Mapped serials with no GPS log:', mapping.set_index('serial').loc[no_gps, 'id'].to_dict())

gps['gps_end'] = pd.to_datetime(gps['endtime'].astype(str).str.rstrip('Z'))
launch = pd.Timestamp(str(cfg.LAUNCHTIME)[:19])
print('\nGPS log ends before launch:')
print(gps.loc[gps['gps_end'] < launch, ['SN', 'seed_id', 'endtime']].to_string(index=False))

Mapped serials with no GPS log: {'361': '1R.B20..HDI'}

GPS log ends before launch:
 SN     seed_id                     endtime
256 1R.B03..HDF 2026-04-01T20:28:53.000000Z
362 1R.B01..HDF 2026-03-14T23:45:55.000000Z


## Offset from the station table (`KSC_seismic_network_2016_2022.xls`, sheet `artemis2`)

In [4]:
stations = pd.read_excel(cfg.STATION_XLS, sheet_name='artemis2')

def offset_m(lat1, lon1, lat2, lon2):
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1) * np.cos(np.radians(0.5 * (lat1 + lat2)))
    return 6371e3 * np.hypot(dlat, dlon)

gps['station'] = gps['seed_id'].str.split('.').str[1]
cmp = gps.merge(stations, on='station', how='left', suffixes=('_gps', '_table'))
cmp['offset_m'] = offset_m(cmp['lat_gps'].astype(float), cmp['lon_gps'].astype(float),
                           cmp['lat_table'], cmp['lon_table']).round(1)
cmp[['SN', 'seed_id', 'lat_gps', 'lon_gps', 'lat_table', 'lon_table', 'offset_m']]

,SN,seed_id,lat_gps,lon_gps,lat_table,lon_table,offset_m
0,244,1R.B14..HDI,28.632276,-80.615963,28.632222,-80.615833,14.0
1,246,1R.B14..HDF,28.632270,-80.615983,28.632222,-80.615833,15.5
2,251,1R.B01..HDI,28.621368,-80.606851,28.621389,-80.606667,18.2
3,254,1R.B03..HDI,28.643006,-80.623604,28.642778,-80.623611,25.4
4,256,1R.B03..HDF,28.642982,-80.623608,28.642778,-80.623611,22.8
5,273,1R.B24LT..HDF,28.580586,-80.642901,28.580500,-80.642900,9.6
6,274,1R.B07.10.HDF,28.607186,-80.594139,28.607222,-80.594167,4.8
7,275,1R.B24..HDI,28.580653,-80.640844,28.580556,-80.640833,10.9
8,285,1R.B20..HDF,28.573620,-80.572249,28.573611,-80.572222,2.8
9,287,1R.B24RE..HDF,28.581152,-80.643291,28.581160,-80.643260,3.2


In [5]:
# Nearest table station to every Gem (identifies the placeholder serial 363, 1R.UNK..HD1)
for _, row in gps.iterrows():
    d = offset_m(float(row['lat']), float(row['lon']), stations['lat'], stations['lon'])
    i = int(np.argmin(d))
    if row['station'] == 'UNK' or d[i] > 50:
        print(f"{row['SN']} {row['seed_id']}: nearest {stations['station'][i]} at {d[i]:.0f} m")

363 1R.UNK..HD1: nearest B20 at 8 m
